In [ ]:
# imports
import numpy as np
import pandas as pd
import scipy.stats as stats
import sys

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')

sys.path.append('helpers/pcca_fa/')
import helpers.pcca_fa.sim_pcca_fa as spf
import helpers.pcca_fa.pcca_fa_mdl as pf
import helpers.pcca_fa.cca.prob_cca as pcca

from dual_pfc_funcs import getParams, load_dict, getBaseSimParams

# params
SAVE_FIG = False
data_path = 'preprocessed_data/'
params = getParams()
subjects, symbols = params['subjects'], params['markers']

In [ ]:
alt_mdls = load_dict(data_path + 'all_alt_mdls_cv.pkl')
pccafa = {}
for sub in subjects:
    pccafa = {**pccafa, **load_dict(data_path + sub + '_pccafa_cv15dim.pkl')}

# compile into single dat with session name as key
dat = {}
for fname in pccafa.keys():
    try:
        dat[fname] = {**alt_mdls[fname], 'pccafa_LL':pccafa[fname]['cvLL']['final_LL'], 'pccafa':pccafa[fname]['params']}
    except: continue

In [ ]:
# data aggregation
df = pd.DataFrame(columns=['SessionID', 'neurons_left', 'neurons_right', 'n_trials', 'NormalizationFactor',
                            'pccafa_LL', 'pcca_LL', 'jointFA_LL', 'indFA_LL', 'pcca_norm_diff', 'jointFA_norm_diff', 'indFA_norm_diff'])


for fname in dat.keys():
    # to compute normalization factor, we need the number of trials and neurons for each session
    n1, n2, N = dat[fname]["n_neurons_left"],dat[fname]["n_neurons_right"],dat[fname]["n_trials"]
    norm_factor = (n1 + n2) * N

    # pCCA-FA LL:
    curr_pccafa_ll = dat[fname]['pccafa_LL']

    # alternative model LLs:
    curr_jfa_ll = np.max(dat[fname]['joint_LL'])
    curr_ifa_ll = np.max(dat[fname]['ind_fa_LH_LL']) + np.max(dat[fname]['ind_fa_RH_LL'])
    curr_pcca_ll = np.max(dat[fname]['pcca_LL'])

    # normalize LL differences:
    pcca_norm_diff = (curr_pccafa_ll - curr_pcca_ll) / norm_factor
    jointFA_norm_diff = (curr_pccafa_ll - curr_jfa_ll) / norm_factor
    indFA_norm_diff = (curr_pccafa_ll - curr_ifa_ll) / norm_factor

    df2 = {'SessionID':fname, 'neurons_left':n1, 'neurons_right':n2, 'n_trials':N, 'NormalizationFactor':norm_factor, 
            'pccafa_LL':curr_pccafa_ll, 'pcca_LL':curr_pcca_ll, 'jointFA_LL':curr_jfa_ll, 'indFA_LL':curr_ifa_ll,
            'pcca_norm_diff':pcca_norm_diff, 'jointFA_norm_diff':jointFA_norm_diff, 'indFA_norm_diff':indFA_norm_diff}
    df.loc[len(df)] = df2
# df

In [ ]:
np.random.seed(3)

fig = plt.figure(constrained_layout=True)
fig.set_figwidth(2*fig.get_figwidth())

box = plt.boxplot(x=[df['pcca_norm_diff'],df['jointFA_norm_diff'],df['indFA_norm_diff']], vert=False, showfliers=False,
            labels=['pCCA', 'joint FA', 'individual FA'])
for _, line_list in box.items():
    for line in line_list: line.set_color('k')

plt.plot([0,0], [0,4], '--', color='k')

s = 12
for sub,sym in zip(subjects,symbols):
    filt = [sub[:2].title() in fname for fname in df['SessionID']]
    tmp_df = df[filt]

    ydata = np.random.uniform(low=1.25, high=1.4, size=len(tmp_df))
    plt.scatter(tmp_df['pcca_norm_diff'], ydata, color='black', marker=sym, alpha=0.5, s=s, zorder=10, edgecolors='none', label='Monkey {}'.format(sub[:2].title()))

    ydata = np.random.uniform(low=2.25, high=2.4, size=len(tmp_df))
    plt.scatter(tmp_df['jointFA_norm_diff'], ydata, color='black', marker=sym, alpha=0.5, s=s, zorder=10, edgecolors='none')

    ydata = np.random.uniform(low=3.25, high=3.4, size=len(tmp_df))
    plt.scatter(tmp_df['indFA_norm_diff'], ydata, color='black', marker=sym, alpha=0.5, s=s, zorder=10, edgecolors='none',)
    
plt.ticklabel_format(axis='x', style='scientific', scilimits=(-2,-2), useMathText=True)
plt.xlabel('Difference in cross-validated data log likelihood (pCCA-FA - alternative model)', labelpad=15)
plt.ylim(0.5,3.75)
plt.legend(markerscale=1.5)

if SAVE_FIG:
    pdf = PdfPages('figs/alt_models.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    plt.show()

In [ ]:
# statistics
alpha = 0.01 
print("Alternative model statistics across sessions:")
for sub in subjects:
    print('Monkey:', sub[:2].title())
    filt = [sub[:2].title() in fname for fname in df['SessionID']]
    tmp_df = df[filt]
    p = stats.ttest_rel(a=tmp_df['pccafa_LL'], b=tmp_df['indFA_LL'],alternative='greater').pvalue
    print("  pCCA-FA > individual FA? {}, p = {:.10f}".format(p < alpha, p))
    p = stats.ttest_rel(a=tmp_df['pccafa_LL'], b=tmp_df['jointFA_LL'],alternative='greater').pvalue
    print("  pCCA-FA > joint FA? {}, p = {:.10f}".format(p < alpha, p))
    p = stats.ttest_rel(a=tmp_df['pccafa_LL'], b=tmp_df['pcca_LL'],alternative='greater').pvalue
    print("  pCCA-FA > pCCA? {}, p = {:.10f}".format(p < alpha, p))

# quantify number of sessions > 0
print()
print("Number of sessions where pCCA-FA > ind FA: {} of {}".format(np.sum(df['indFA_norm_diff'] > 0), len(df)))
print("Number of sessions where pCCA-FA > joint FA: {} of {}".format(np.sum(df['jointFA_norm_diff'] > 0), len(df)))
print("Number of sessions where pCCA-FA > pCCA: {} of {}".format(np.sum(df['pcca_norm_diff'] > 0), len(df)))

In [ ]:
# illustrate parameter counts
seed = 10

p = getBaseSimParams()
n_trials = p['n_trials']
n1,n2 = 10, 10
d,d1,d2 = p['d'], p['d1'], p['d2']
sv_goal = (30,30)
    
base_simulator = spf.sim_pcca_fa(n1,n2,d,d1,d2,sv_goal=sv_goal,rand_seed=seed)
base_params = base_simulator.get_params()
X_1,X_2 = base_simulator.sim_data(n_trials)

# get sample cov
mu_x1,mu_x2 = X_1.mean(axis=0),X_2.mean(axis=0)
X_1c,X_2c = (X_1-mu_x1), (X_2-mu_x2) 
X_total = np.concatenate((X_1c,X_2c),axis=1)
sampleCov = 1/n_trials * (X_total.T).dot(X_total)

# fit the data using pcca-fa
model = pf.pcca_fa()
model.train(X_1,X_2,d,d1,d2,parallelize=True,rand_seed=seed)
est_params = model.get_params()
L = est_params['L_total']
pccafa_shared = L @ L.T
pccafa_shared_across = L[:, :d] @ L[:, :d].T
pccafa_shared_within = L[:, d:] @ L[:, d:].T
Psi_total = np.zeros((n1+n2, n1+n2))
Psi_total[:n1,:n1] = np.diag(est_params['psi_1'])
Psi_total[n1:,n1:] = np.diag(est_params['psi_2'])

# fit the data using pcca
model = pcca.prob_cca()
model.train(X_1,X_2,d,rand_seed=seed)
est_params = model.get_params()
pcca_shared = est_params['L_total'] @ est_params['L_total'].T
Psi_pcca = np.zeros((n1+n2, n1+n2))
Psi_pcca[:n1,:n1] = est_params['psi_x']
Psi_pcca[n1:,n1:] = est_params['psi_y']

# find min and max vals for colorbars
all_params = np.concatenate((pccafa_shared,pcca_shared,Psi_total,Psi_pcca,pccafa_shared_within,pccafa_shared_across),axis=None)
max_val = all_params.max()
min_val = all_params.min()

# sample covariance matrix
fig, ax = plt.subplots()
ax.imshow(sampleCov, cmap='gray')
ax.invert_yaxis()
ax.set_axis_off()

if SAVE_FIG:
    pdf = PdfPages('figs/sim_sample_cov.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    plt.show()

# pcca-fa parameters
fig, ax = plt.subplots(1,3)
ax[0].imshow(pccafa_shared_across, cmap='gray', vmin=min_val, vmax=max_val)
ax[0].invert_yaxis()
ax[0].set_axis_off()
ax[1].imshow(pccafa_shared_within, cmap='gray', vmin=min_val, vmax=max_val)
ax[1].invert_yaxis()
ax[1].set_axis_off()
ax[2].imshow(Psi_total, cmap='gray', vmin=min_val, vmax=max_val)
ax[2].invert_yaxis()
ax[2].set_axis_off()

if SAVE_FIG:
    pdf = PdfPages('figs/sim_pccafa_params.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    plt.show()

# pcca parameters
fig, ax = plt.subplots(1,2)
ax[0].imshow(pcca_shared, cmap='gray', vmin=min_val, vmax=max_val)
ax[0].invert_yaxis()
ax[0].set_axis_off()
im = ax[1].imshow(Psi_pcca, cmap='gray', vmin=min_val, vmax=max_val)
ax[1].invert_yaxis()
ax[1].set_axis_off()

fig.colorbar(im, ax=ax[1])

if SAVE_FIG:
    pdf = PdfPages('figs/sim_pcca_params.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    plt.show()